# Phase 4 Exp4 4(a): feature-assignment UNI official test

Evaluate the **15 frozen** feature-assignment checkpoints (3 seeds × 5 folds) on the 129 official CAMELYON16 test slides. The model is not retrained. Per-slide probabilities are averaged across all 15 checkpoints. The classification threshold is fixed from the model’s training-slide OOF predictions using Youden J.

Attach two Kaggle inputs:
1. The prepared CAMELYON16-UNI dataset with `PREPROCESSING_COMPLETE.json`, `slides.csv`, `bags/`, and `coords/`.
2. The saved Phase-4 Exp4 output containing the complete `exp4_4a_feature_assignment/` folder, including all 15 `checkpoint.pt` files.

The notebook embeds audited official-test patch-width metadata from the project’s Phase-4 γ=1.0 test reference; no third Kaggle input is needed. The metadata is used only to scale coordinates, not to train or select a model. Enable a GPU, then choose **Save Version → Save & Run All**. Outputs are written to `/kaggle/working/feature-assignment-uni-official-test/`.

**Variant detail:** the saved `config.json` incorrectly lists spatial assignment weight 0.5 for this variant. Its hash-verified training runner actually uses weight **0.0**; this notebook reproduces that feature-only rule.


In [ ]:
%%writefile /kaggle/working/flat_topk_uni_official_test.py
"""Evaluate the 15 frozen Phase-4 flat Top-K UNI checkpoints on CAMELYON16 test.

Authors: Srimanreddy (DSMIL baseline), Hrishikesh Kurapati (Phase-4 experiments).
The notebook `flat_topk_uni_official_test.ipynb` invokes this standalone script.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, roc_auc_score, roc_curve)
from torch import nn


SEEDS = (42, 43, 44)
FOLDS = tuple(range(5))
EXPECTED_RUNNER_SHA256 = "faeba55515d5c92d66de5714d9346f59b0851725e89b24c444751aa35e190d22"


class FlatTopKUNI(nn.Module):
    """Inference-equivalent to phase4_exp3_4_runner.FlatTopKUNI."""

    def __init__(self):
        super().__init__()
        self.i_classifier = nn.Module()
        self.i_classifier.fc = nn.Sequential(nn.Linear(1024, 1))
        self.b_classifier = nn.Module()
        self.b_classifier.q = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(),
                                            nn.Linear(128, 128), nn.Tanh())
        self.b_classifier.v = nn.Identity()
        self.b_classifier.fcc = nn.Conv1d(1, 1, kernel_size=1024)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scores = self.i_classifier.fc(x)[:, 0]
        values = scores.detach().cpu().numpy()
        order = np.lexsort((np.arange(len(values)), -values))[:min(4, len(values))]
        index = torch.as_tensor(order.copy(), device=x.device)
        q = self.b_classifier.q(x)
        attention = torch.softmax((q @ q[index].T) / math.sqrt(128), dim=0)
        anchor_bags = attention.T @ self.b_classifier.v(x)
        bag = anchor_bags.mean(dim=0).view(1, 1, 1024)
        return self.b_classifier.fcc(bag).view(())


def read_csv(path: Path) -> list[dict]:
    with path.open(newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def discover(inputs: Path) -> tuple[Path, Path]:
    prepared = []
    for marker in inputs.rglob("PREPROCESSING_COMPLETE.json"):
        info = json.loads(marker.read_text())
        if info.get("source") == "uni" and info.get("feature_dim") == 1024:
            prepared.append(marker.parent)
    references = []
    for path in inputs.rglob("cv_summary.json"):
        if json.loads(path.read_text()).get("variant") == "exp4_flat_topk_uni":
            references.append(path.parent)
    if len(prepared) != 1 or len(references) != 1:
        raise ValueError(f"Attach one prepared UNI dataset and one flat Top-K output: {prepared}, {references}")
    return prepared[0], references[0]


def validate(prepared: Path, reference: Path) -> tuple[list[dict], float]:
    cfg = json.loads((reference / "config.json").read_text())
    cv = json.loads((reference / "cv_summary.json").read_text())
    complete = json.loads((reference / "COMPLETE.json").read_text())
    info = json.loads((prepared / "PREPROCESSING_COMPLETE.json").read_text())
    if cfg.get("variant") != cv.get("variant") or cv.get("variant") != "exp4_flat_topk_uni":
        raise ValueError("Wrong checkpoint variant")
    if cfg.get("runner_sha256") != EXPECTED_RUNNER_SHA256:
        raise ValueError("Checkpoint runner differs from the verified model implementation")
    if cfg.get("prepared_manifest_sha256") != sha256(prepared / "slides.csv"):
        raise ValueError("Checkpoint/UNI manifest mismatch")
    if cfg.get("seeds") != list(SEEDS) or cfg.get("folds") != list(FOLDS):
        raise ValueError("Unexpected seed or fold protocol")
    if len(cv.get("runs", [])) != 15 or complete.get("fits") != 15 or complete.get("test_evaluated") is not False:
        raise ValueError("Incomplete validation reference")
    if any(info.get(k) != v for k, v in {"source": "uni", "feature_dim": 1024,
                                         "slides": 399, "train_slides": 270,
                                         "test_slides": 129}.items()):
        raise ValueError("Unexpected prepared UNI dataset")
    if any(not (reference / f"seed_{s}" / f"fold_{f}" / "checkpoint.pt").is_file()
           for s in SEEDS for f in FOLDS):
        raise FileNotFoundError("Attach the complete output with all 15 checkpoint.pt files")
    rows = read_csv(prepared / "slides.csv")
    train = [r for r in rows if r["official_split"] == "train"]
    test = [r for r in rows if r["official_split"] == "test"]
    if len(rows) != 399 or len(train) != 270 or len(test) != 129 or len({r["slide_id"] for r in rows}) != 399:
        raise ValueError("UNI slide manifest has incorrect coverage")
    if next(r["label"] for r in test if r["slide_id"] == "test_114") != "0":
        raise ValueError("Unexpected test_114 label")
    oof = read_csv(reference / "oof_predictions_ensemble.csv")
    by_id = {r["slide_id"]: r for r in train}
    if len(oof) != 270 or {r["slide_id"] for r in oof} != set(by_id):
        raise ValueError("Incomplete OOF predictions")
    if any(int(r["label"]) != int(by_id[r["slide_id"]]["label"]) or
           int(r["fold"]) != int(by_id[r["slide_id"]]["validation_fold"]) for r in oof):
        raise ValueError("OOF labels or fold assignments do not match")
    y = np.array([int(r["label"]) for r in oof])
    prob = np.array([float(r["probability"]) for r in oof])
    if not math.isclose(roc_auc_score(y, prob), cv["oof_ensemble_auc"], abs_tol=1e-12):
        raise ValueError("OOF AUC does not match saved CV summary")
    fpr, tpr, thresholds = roc_curve(y, prob)
    finite = np.flatnonzero(np.isfinite(thresholds))
    threshold = float(thresholds[finite[np.argmax((tpr - fpr)[finite])]])
    return test, threshold


def evaluate(prepared: Path, reference: Path, output: Path, test: list[dict], threshold: float) -> dict:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = []
    checkpoints = {}
    for seed in SEEDS:
        for fold in FOLDS:
            path = reference / f"seed_{seed}" / f"fold_{fold}" / "checkpoint.pt"
            model = FlatTopKUNI().to(device)
            model.load_state_dict(torch.load(path, map_location=device, weights_only=True), strict=True)
            model.eval()
            models.append((seed, fold, model))
            checkpoints[f"seed_{seed}/fold_{fold}/checkpoint.pt"] = sha256(path)
    per_checkpoint, ensemble = [], []
    with torch.inference_mode():
        for index, row in enumerate(sorted(test, key=lambda r: r["slide_id"]), 1):
            path = prepared / row["bag_path"]
            x = np.load(path, mmap_mode="r", allow_pickle=False)
            if x.shape != (int(row["n_patches"]), 1024) or x.dtype != np.float32 or not np.isfinite(x).all():
                raise ValueError(f"Invalid UNI test bag: {row['slide_id']}")
            features = torch.tensor(x, dtype=torch.float32, device=device)
            probs = []
            for seed, fold, model in models:
                p = float(torch.sigmoid(model(features)))
                if not math.isfinite(p):
                    raise ValueError(f"Nonfinite probability: {row['slide_id']}")
                probs.append(p)
                per_checkpoint.append({"seed": seed, "fold": fold, "slide_id": row["slide_id"],
                                       "label": int(row["label"]), "probability": p})
            avg = float(np.mean(probs))
            ensemble.append({"slide_id": row["slide_id"], "label": int(row["label"]),
                             "probability": avg, "prediction": int(avg >= threshold)})
            print(f"{index:3d}/129 {row['slide_id']} ensemble_probability={avg:.6f}", flush=True)
            del features
    if len(per_checkpoint) != 1935 or len(ensemble) != 129:
        raise ValueError("Incomplete official-test predictions")
    output.mkdir(parents=True, exist_ok=True)
    write_csv(output / "test_predictions_per_checkpoint.csv", per_checkpoint,
              ["seed", "fold", "slide_id", "label", "probability"])
    write_csv(output / "test_predictions_ensemble.csv", ensemble,
              ["slide_id", "label", "probability", "prediction"])
    y = [r["label"] for r in ensemble]
    pred = [r["prediction"] for r in ensemble]
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    summary = {"model": "Phase-4 flat Top-K UNI", "test_slides": 129, "n_models": 15,
               "test_auc": float(roc_auc_score(y, [r["probability"] for r in ensemble])),
               "test_accuracy": float(accuracy_score(y, pred)),
               "test_balanced_accuracy": float(balanced_accuracy_score(y, pred)),
               "test_sensitivity": float(tp / (tp + fn)),
               "test_specificity": float(tn / (tn + fp)),
               "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
               "threshold_from_oof": threshold,
               "ensemble_rule": "mean of 15 frozen checkpoint bag probabilities",
               "prepared_manifest_sha256": sha256(prepared / "slides.csv"),
               "checkpoint_sha256": checkpoints}
    (output / "test_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps({k: v for k, v in summary.items() if k != "checkpoint_sha256"}, indent=2))
    return summary


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--output-root", type=Path, default=Path("/kaggle/working/flat-topk-uni-official-test"))
    args = parser.parse_args()
    prepared, reference = discover(args.input_root)
    test, threshold = validate(prepared, reference)
    print("Prepared:", prepared, "\nFrozen checkpoints:", reference,
          "\nOOF threshold:", threshold, flush=True)
    evaluate(prepared, reference, args.output_root, test, threshold)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/feature_assignment_uni_official_test.py
"""Official test inference for frozen Phase-4 Exp4 feature-assignment checkpoints.

Authors: Srimanreddy (DSMIL baseline), Hrishikesh Kurapati (Phase-4 experiments).
The paired notebook embeds this file and audited official-test coordinate metadata.
"""

from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, roc_auc_score, roc_curve)

import flat_topk_uni_official_test as common


def hierarchy_indices(coords: torch.Tensor, x: torch.Tensor, anchors: list[int], grid: int = 4):
    """Exact feature-only, anchor-dependent partition from the validated runner."""
    xy = coords.detach().cpu().numpy()
    features = x.detach().cpu().numpy()
    cell = np.floor(xy / grid).astype(np.int64)
    per_cell: dict[tuple[int, int], list[int]] = {}
    for i, key in enumerate(cell):
        per_cell.setdefault(tuple(key), []).append(i)
    groups = []
    for key, members in sorted(per_cell.items()):
        present = [a for a in anchors if a in members]
        if len(present) <= 1:
            groups.append((key, members, present[0] if present else None))
        else:
            partitions = {a: [] for a in present}
            for i in members:
                owner = (i if i in partitions else
                         min(present, key=lambda a: (np.linalg.norm(xy[i]-xy[a]), anchors.index(a))))
                partitions[owner].append(i)
            groups.extend((key, rows, a) for a, rows in partitions.items())
    branch_groups: list[list[int]] = [[] for _ in anchors]
    for group_index, (key, members, own_anchor) in enumerate(groups):
        if own_anchor is not None:
            branch = anchors.index(own_anchor)
        else:
            mean_feature = features[members].mean(axis=0)
            mean_feature = mean_feature / max(np.linalg.norm(mean_feature), 1e-12)
            ratings = []
            for a in anchors:
                anchor_feature = features[a] / max(np.linalg.norm(features[a]), 1e-12)
                ratings.append(0.5 * float(mean_feature @ anchor_feature))
            branch = int(np.argmax(ratings))
        branch_groups[branch].append(group_index)
    if any(not group for group in branch_groups):
        raise ValueError("Anchor branch has no microgroup")
    regions: dict[tuple[int, int, int], list[int]] = {}
    group_branch = {}
    for branch, members in enumerate(branch_groups):
        for gi in members:
            group_branch[gi] = branch
            key = groups[gi][0]
            regions.setdefault((branch, key[0] // 2, key[1] // 2), []).append(gi)
    region_list = [regions[key] for key in sorted(regions)]
    region_branch = [key[0] for key in sorted(regions)]
    parent = {gi: region for region, child in enumerate(region_list) for gi in child}
    assert sorted(i for _, members, _ in groups for i in members) == list(range(len(x)))
    return groups, branch_groups, region_list, region_branch, parent, group_branch


class HierarchicalFeatureAssignment(common.FlatTopKUNI):
    """Exp4 4(a), level 1, space_weight=0, grid width 4."""

    def pool(self, children: torch.Tensor, segment: torch.Tensor, anchors: torch.Tensor):
        count = len(anchors)
        query = self.b_classifier.q(children)
        anchor_query = self.b_classifier.q(anchors)
        logits = (query * anchor_query[segment]).sum(dim=1) / math.sqrt(128)
        maxima = logits.new_full((count,), -torch.inf).scatter_reduce(
            0, segment, logits, reduce="amax", include_self=True)
        exp = torch.exp(logits - maxima[segment])
        denominator = logits.new_zeros(count).index_add(0, segment, exp)
        weights = exp / denominator[segment]
        return children.new_zeros((count, children.shape[1])).index_add(
            0, segment, children * weights[:, None])

    def forward(self, x: torch.Tensor, coords: torch.Tensor) -> torch.Tensor:
        scores = self.i_classifier.fc(x)[:, 0]
        values = scores.detach().cpu().numpy()
        order = np.lexsort((np.arange(len(values)), -values))[:min(4, len(values))].tolist()
        anchors = x[order]
        groups, branch_groups, regions, region_branch, parent, group_branch = hierarchy_indices(
            coords, x, order, 4)
        patch_group = torch.empty(len(x), dtype=torch.long, device=x.device)
        for gi, (_, members, _) in enumerate(groups):
            patch_group[torch.as_tensor(members, device=x.device)] = gi
        group_branch_t = torch.as_tensor([group_branch[gi] for gi in range(len(groups))],
                                         dtype=torch.long, device=x.device)
        parent_t = torch.as_tensor([parent[gi] for gi in range(len(groups))],
                                   dtype=torch.long, device=x.device)
        region_branch_t = torch.as_tensor(region_branch, dtype=torch.long, device=x.device)
        group_vectors = self.pool(x, patch_group, anchors[group_branch_t])
        region_vectors = self.pool(group_vectors, parent_t, anchors[region_branch_t])
        root_vectors = self.pool(region_vectors, region_branch_t, anchors)
        return self.b_classifier.fcc(root_vectors.mean(dim=0).view(1, 1, 1024)).view(())


def discover(inputs: Path) -> tuple[Path, Path]:
    prepared = []
    for marker in inputs.rglob("PREPROCESSING_COMPLETE.json"):
        info = json.loads(marker.read_text())
        if info.get("source") == "uni" and info.get("feature_dim") == 1024:
            prepared.append(marker.parent)
    references = []
    for path in inputs.rglob("cv_summary.json"):
        if json.loads(path.read_text()).get("variant") == "exp4_4a_feature_assignment":
            references.append(path.parent)
    if len(prepared) != 1 or len(references) != 1:
        raise ValueError(f"Attach one prepared UNI dataset and one Exp4 feature assignment output: {prepared}, {references}")
    return prepared[0], references[0]


def validate(prepared: Path, reference: Path, metadata_path: Path):
    cfg = json.loads((reference / "config.json").read_text())
    cv = json.loads((reference / "cv_summary.json").read_text())
    complete = json.loads((reference / "COMPLETE.json").read_text())
    info = json.loads((prepared / "PREPROCESSING_COMPLETE.json").read_text())
    if cfg.get("variant") != cv.get("variant") or cv.get("variant") != "exp4_4a_feature_assignment":
        raise ValueError("Wrong checkpoint variant")
    if cfg.get("runner_sha256") != common.EXPECTED_RUNNER_SHA256:
        raise ValueError("Checkpoint runner differs from the verified model implementation")
    if cfg.get("grid_width_patch_units") != 4:
        raise ValueError("Unexpected hierarchy grid width")
    # The saved config incorrectly lists 0.5 as this variant's spatial weight.
    # The hash-verified runner instantiates this named variant with weight 0.0.
    if cfg.get("prepared_manifest_sha256") != common.sha256(prepared / "slides.csv"):
        raise ValueError("Checkpoint/UNI manifest mismatch")
    if cfg.get("seeds") != list(common.SEEDS) or cfg.get("folds") != list(common.FOLDS):
        raise ValueError("Unexpected seed or fold protocol")
    if len(cv.get("runs", [])) != 15 or complete.get("fits") != 15 or complete.get("test_evaluated") is not False:
        raise ValueError("Incomplete validation reference")
    if any(info.get(k) != v for k, v in {"source": "uni", "feature_dim": 1024,
                                         "slides": 399, "train_slides": 270,
                                         "test_slides": 129}.items()):
        raise ValueError("Unexpected prepared UNI dataset")
    if any(not (reference / f"seed_{s}" / f"fold_{f}" / "checkpoint.pt").is_file()
           for s in common.SEEDS for f in common.FOLDS):
        raise FileNotFoundError("Attach all 15 feature-assignment checkpoints")
    rows = common.read_csv(prepared / "slides.csv")
    train = [r for r in rows if r["official_split"] == "train"]
    test = [r for r in rows if r["official_split"] == "test"]
    if len(rows) != 399 or len(train) != 270 or len(test) != 129 or len({r["slide_id"] for r in rows}) != 399:
        raise ValueError("UNI slide manifest has incorrect coverage")
    if next(r["label"] for r in test if r["slide_id"] == "test_114") != "0":
        raise ValueError("Unexpected test_114 label")
    metadata = json.loads(metadata_path.read_text())
    items = metadata["slides"]
    by_id = {r["slide_id"]: r for r in items}
    if len(items) != 129 or len(by_id) != 129 or set(by_id) != {r["slide_id"] for r in test}:
        raise ValueError("Incomplete audited test coordinate metadata")
    for row in test:
        item = by_id[row["slide_id"]]
        n = int(row["n_patches"])
        if (item["patches"] != n or item["coordinate_shape"] != [n, 2] or
                item["patch_level"] != 0 or item["patch_size_px"] <= 0 or
                len(item["level_dim"]) != 2 or min(item["level_dim"]) <= 0 or
                item["out_of_bounds_coordinate_rows"] != 0):
            raise ValueError(f"Invalid test coordinate metadata: {row['slide_id']}")
    oof = common.read_csv(reference / "oof_predictions_ensemble.csv")
    by_train = {r["slide_id"]: r for r in train}
    if len(oof) != 270 or {r["slide_id"] for r in oof} != set(by_train):
        raise ValueError("Incomplete OOF predictions")
    if any(int(r["label"]) != int(by_train[r["slide_id"]]["label"]) or
           int(r["fold"]) != int(by_train[r["slide_id"]]["validation_fold"]) for r in oof):
        raise ValueError("OOF labels or folds differ from prepared UNI")
    y = np.array([int(r["label"]) for r in oof])
    p = np.array([float(r["probability"]) for r in oof])
    if not math.isclose(roc_auc_score(y, p), cv["oof_ensemble_auc"], abs_tol=1e-12):
        raise ValueError("OOF AUC does not match the saved summary")
    fpr, tpr, thresholds = roc_curve(y, p)
    finite = np.flatnonzero(np.isfinite(thresholds))
    threshold = float(thresholds[finite[np.argmax((tpr - fpr)[finite])]])
    return test, by_id, threshold


def evaluate(prepared: Path, reference: Path, output: Path, test: list[dict], metadata: dict, threshold: float):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = []
    hashes = {}
    for seed in common.SEEDS:
        for fold in common.FOLDS:
            path = reference / f"seed_{seed}" / f"fold_{fold}" / "checkpoint.pt"
            model = HierarchicalFeatureAssignment().to(device)
            model.load_state_dict(torch.load(path, map_location=device, weights_only=True), strict=True)
            model.eval()
            models.append((seed, fold, model))
            hashes[f"seed_{seed}/fold_{fold}/checkpoint.pt"] = common.sha256(path)
    per_checkpoint, ensemble = [], []
    with torch.inference_mode():
        for index, row in enumerate(sorted(test, key=lambda r: r["slide_id"]), 1):
            n = int(row["n_patches"])
            bag = np.load(prepared / row["bag_path"], mmap_mode="r", allow_pickle=False)
            raw = np.load(prepared / row["coords_path"], mmap_mode="r", allow_pickle=False)
            item = metadata[row["slide_id"]]
            if (bag.shape != (n, 1024) or bag.dtype != np.float32 or
                    raw.shape != (n, 2) or raw.dtype != np.int64 or
                    not np.isfinite(bag).all() or np.any(raw < 0) or
                    np.any(raw >= np.asarray(item["level_dim"]))):
                raise ValueError(f"Invalid UNI test arrays: {row['slide_id']}")
            features = torch.tensor(bag, dtype=torch.float32, device=device)
            coords = torch.tensor(np.asarray(raw, dtype=np.float32) / item["patch_size_px"],
                                  dtype=torch.float32, device=device)
            probs = []
            for seed, fold, model in models:
                p = float(torch.sigmoid(model(features, coords)))
                if not math.isfinite(p):
                    raise ValueError(f"Nonfinite probability: {row['slide_id']}")
                probs.append(p)
                per_checkpoint.append({"seed": seed, "fold": fold, "slide_id": row["slide_id"],
                                       "label": int(row["label"]), "probability": p})
            avg = float(np.mean(probs))
            ensemble.append({"slide_id": row["slide_id"], "label": int(row["label"]),
                             "probability": avg, "prediction": int(avg >= threshold)})
            print(f"{index:3d}/129 {row['slide_id']} ensemble_probability={avg:.6f}", flush=True)
            del features, coords
    if len(per_checkpoint) != 1935 or len(ensemble) != 129:
        raise ValueError("Incomplete official-test predictions")
    output.mkdir(parents=True, exist_ok=True)
    common.write_csv(output / "test_predictions_per_checkpoint.csv", per_checkpoint,
                     ["seed", "fold", "slide_id", "label", "probability"])
    common.write_csv(output / "test_predictions_ensemble.csv", ensemble,
                     ["slide_id", "label", "probability", "prediction"])
    y = [r["label"] for r in ensemble]
    pred = [r["prediction"] for r in ensemble]
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    summary = {"model": "Phase-4 Exp4 4(a) feature assignment", "test_slides": 129,
               "n_models": 15, "test_auc": float(roc_auc_score(y, [r["probability"] for r in ensemble])),
               "test_accuracy": float(accuracy_score(y, pred)),
               "test_balanced_accuracy": float(balanced_accuracy_score(y, pred)),
               "test_sensitivity": float(tp / (tp + fn)),
               "test_specificity": float(tn / (tn + fp)),
               "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
               "threshold_from_oof": threshold,
               "ensemble_rule": "mean of 15 frozen checkpoint bag probabilities",
               "assignment_spatial_weight_actual": 0.0,
               "prepared_manifest_sha256": common.sha256(prepared / "slides.csv"),
               "test_coordinate_metadata_sha256": common.sha256(Path(__file__).parent / "test_coordinate_metadata.json"),
               "checkpoint_sha256": hashes}
    (output / "test_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps({k: v for k, v in summary.items() if k != "checkpoint_sha256"}, indent=2))
    return summary


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--metadata", type=Path,
                        default=Path("/kaggle/working/test_coordinate_metadata.json"))
    parser.add_argument("--output-root", type=Path,
                        default=Path("/kaggle/working/feature-assignment-uni-official-test"))
    args = parser.parse_args()
    prepared, reference = discover(args.input_root)
    test, metadata, threshold = validate(prepared, reference, args.metadata)
    print("Prepared:", prepared, "\nFrozen checkpoints:", reference,
          "\nOOF threshold:", threshold, flush=True)
    evaluate(prepared, reference, args.output_root, test, metadata, threshold)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/test_coordinate_metadata.json
{"source_audit_sha256":"1b52ddca2d9c3e0e6ddef4a16712e1f948805ef87a222569fc2115e97b522785","slides":[{"slide_id":"test_001","patches":14198,"coordinate_shape":[14198,2],"level_dim":[86016,89600],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_002","patches":7995,"coordinate_shape":[7995,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_003","patches":6254,"coordinate_shape":[6254,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_004","patches":14656,"coordinate_shape":[14656,2],"level_dim":[98304,103936],"patch_size_px":564,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_005","patches":3672,"coordinate_shape":[3672,2],"level_dim":[97792,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_006","patches":9586,"coordinate_shape":[9586,2],"level_dim":[97792,217088],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_007","patches":5447,"coordinate_shape":[5447,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_008","patches":15384,"coordinate_shape":[15384,2],"level_dim":[106496,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_009","patches":6501,"coordinate_shape":[6501,2],"level_dim":[97792,216064],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_010","patches":14883,"coordinate_shape":[14883,2],"level_dim":[86528,207360],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_011","patches":13221,"coordinate_shape":[13221,2],"level_dim":[98816,199680],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_012","patches":18264,"coordinate_shape":[18264,2],"level_dim":[126976,78848],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_013","patches":3381,"coordinate_shape":[3381,2],"level_dim":[96768,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_014","patches":21749,"coordinate_shape":[21749,2],"level_dim":[110592,100352],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_015","patches":2849,"coordinate_shape":[2849,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_016","patches":19976,"coordinate_shape":[19976,2],"level_dim":[110592,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_017","patches":5520,"coordinate_shape":[5520,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_018","patches":7946,"coordinate_shape":[7946,2],"level_dim":[81920,75264],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_019","patches":27439,"coordinate_shape":[27439,2],"level_dim":[184320,96768],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_020","patches":1373,"coordinate_shape":[1373,2],"level_dim":[96768,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_021","patches":16201,"coordinate_shape":[16201,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_022","patches":15188,"coordinate_shape":[15188,2],"level_dim":[131072,75264],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_023","patches":7563,"coordinate_shape":[7563,2],"level_dim":[97792,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_024","patches":4113,"coordinate_shape":[4113,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_025","patches":1955,"coordinate_shape":[1955,2],"level_dim":[97792,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_026","patches":18711,"coordinate_shape":[18711,2],"level_dim":[98304,103936],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_027","patches":6801,"coordinate_shape":[6801,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_028","patches":14187,"coordinate_shape":[14187,2],"level_dim":[155648,78848],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_029","patches":3290,"coordinate_shape":[3290,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_030","patches":6933,"coordinate_shape":[6933,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_031","patches":12804,"coordinate_shape":[12804,2],"level_dim":[86016,103936],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_032","patches":11334,"coordinate_shape":[11334,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_033","patches":39947,"coordinate_shape":[39947,2],"level_dim":[188416,103936],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_034","patches":10248,"coordinate_shape":[10248,2],"level_dim":[155648,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_035","patches":7579,"coordinate_shape":[7579,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_036","patches":6608,"coordinate_shape":[6608,2],"level_dim":[97792,221696],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_037","patches":31244,"coordinate_shape":[31244,2],"level_dim":[122880,111104],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_038","patches":5895,"coordinate_shape":[5895,2],"level_dim":[97792,218624],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_039","patches":1856,"coordinate_shape":[1856,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_040","patches":4470,"coordinate_shape":[4470,2],"level_dim":[97792,221696],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_041","patches":15569,"coordinate_shape":[15569,2],"level_dim":[102400,103936],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_042","patches":28370,"coordinate_shape":[28370,2],"level_dim":[135168,107520],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_043","patches":3447,"coordinate_shape":[3447,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_044","patches":9906,"coordinate_shape":[9906,2],"level_dim":[90112,75264],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_045","patches":4058,"coordinate_shape":[4058,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_046","patches":3689,"coordinate_shape":[3689,2],"level_dim":[61440,89600],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_047","patches":8152,"coordinate_shape":[8152,2],"level_dim":[81920,86016],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_048","patches":19098,"coordinate_shape":[19098,2],"level_dim":[98304,82432],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_050","patches":10351,"coordinate_shape":[10351,2],"level_dim":[97792,214016],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_051","patches":5351,"coordinate_shape":[5351,2],"level_dim":[96768,211968],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_052","patches":7189,"coordinate_shape":[7189,2],"level_dim":[97792,219648],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_053","patches":15012,"coordinate_shape":[15012,2],"level_dim":[97792,218624],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_054","patches":9011,"coordinate_shape":[9011,2],"level_dim":[97792,217088],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_055","patches":6696,"coordinate_shape":[6696,2],"level_dim":[94208,35840],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_056","patches":16786,"coordinate_shape":[16786,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_057","patches":3441,"coordinate_shape":[3441,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_058","patches":15747,"coordinate_shape":[15747,2],"level_dim":[94208,100352],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_059","patches":6969,"coordinate_shape":[6969,2],"level_dim":[97792,217088],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_060","patches":3062,"coordinate_shape":[3062,2],"level_dim":[92672,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_061","patches":8729,"coordinate_shape":[8729,2],"level_dim":[97792,216064],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_062","patches":5470,"coordinate_shape":[5470,2],"level_dim":[135168,28672],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_063","patches":13344,"coordinate_shape":[13344,2],"level_dim":[102400,78848],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_064","patches":1808,"coordinate_shape":[1808,2],"level_dim":[97792,212992],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_065","patches":3347,"coordinate_shape":[3347,2],"level_dim":[77824,53760],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_066","patches":19493,"coordinate_shape":[19493,2],"level_dim":[155648,86016],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_067","patches":11313,"coordinate_shape":[11313,2],"level_dim":[97280,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_068","patches":14549,"coordinate_shape":[14549,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_069","patches":46492,"coordinate_shape":[46492,2],"level_dim":[212992,103936],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_070","patches":4547,"coordinate_shape":[4547,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_071","patches":22076,"coordinate_shape":[22076,2],"level_dim":[155648,96768],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_072","patches":4060,"coordinate_shape":[4060,2],"level_dim":[97792,211456],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_073","patches":13831,"coordinate_shape":[13831,2],"level_dim":[159744,86016],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_074","patches":10912,"coordinate_shape":[10912,2],"level_dim":[97792,211968],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_075","patches":8644,"coordinate_shape":[8644,2],"level_dim":[97792,212992],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_076","patches":12546,"coordinate_shape":[12546,2],"level_dim":[97792,217088],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_077","patches":14338,"coordinate_shape":[14338,2],"level_dim":[126976,75264],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_078","patches":15162,"coordinate_shape":[15162,2],"level_dim":[110592,96768],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_079","patches":10181,"coordinate_shape":[10181,2],"level_dim":[102400,86016],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_080","patches":17312,"coordinate_shape":[17312,2],"level_dim":[122880,96768],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_081","patches":6466,"coordinate_shape":[6466,2],"level_dim":[98816,199680],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_082","patches":11497,"coordinate_shape":[11497,2],"level_dim":[97792,217088],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_083","patches":2164,"coordinate_shape":[2164,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_084","patches":5695,"coordinate_shape":[5695,2],"level_dim":[118784,82432],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_085","patches":3689,"coordinate_shape":[3689,2],"level_dim":[97792,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_086","patches":10810,"coordinate_shape":[10810,2],"level_dim":[86016,71680],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_087","patches":5099,"coordinate_shape":[5099,2],"level_dim":[96768,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_088","patches":4798,"coordinate_shape":[4798,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_089","patches":9609,"coordinate_shape":[9609,2],"level_dim":[135168,64512],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_090","patches":18183,"coordinate_shape":[18183,2],"level_dim":[97792,214528],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_091","patches":3840,"coordinate_shape":[3840,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_092","patches":22061,"coordinate_shape":[22061,2],"level_dim":[126976,103936],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_093","patches":4769,"coordinate_shape":[4769,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_094","patches":13821,"coordinate_shape":[13821,2],"level_dim":[94208,82432],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_095","patches":11042,"coordinate_shape":[11042,2],"level_dim":[110592,57344],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_096","patches":1963,"coordinate_shape":[1963,2],"level_dim":[97792,218624],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_097","patches":11326,"coordinate_shape":[11326,2],"level_dim":[97280,189952],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_098","patches":13933,"coordinate_shape":[13933,2],"level_dim":[131072,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_099","patches":15921,"coordinate_shape":[15921,2],"level_dim":[98304,86016],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_100","patches":18842,"coordinate_shape":[18842,2],"level_dim":[97792,215552],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_101","patches":18287,"coordinate_shape":[18287,2],"level_dim":[176128,57344],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_102","patches":15486,"coordinate_shape":[15486,2],"level_dim":[192512,111104],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_103","patches":5164,"coordinate_shape":[5164,2],"level_dim":[97792,205824],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_104","patches":11187,"coordinate_shape":[11187,2],"level_dim":[96768,210432],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_105","patches":24943,"coordinate_shape":[24943,2],"level_dim":[126976,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_106","patches":4439,"coordinate_shape":[4439,2],"level_dim":[97792,219648],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_107","patches":1872,"coordinate_shape":[1872,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_108","patches":5783,"coordinate_shape":[5783,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_109","patches":16813,"coordinate_shape":[16813,2],"level_dim":[167936,57344],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_110","patches":12284,"coordinate_shape":[12284,2],"level_dim":[102400,89600],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_111","patches":9417,"coordinate_shape":[9417,2],"level_dim":[97792,220672],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_112","patches":4718,"coordinate_shape":[4718,2],"level_dim":[97792,214528],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_113","patches":20317,"coordinate_shape":[20317,2],"level_dim":[131072,107520],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_114","patches":14367,"coordinate_shape":[14367,2],"level_dim":[110592,96768],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_115","patches":1692,"coordinate_shape":[1692,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_116","patches":19685,"coordinate_shape":[19685,2],"level_dim":[87552,71168],"patch_size_px":563,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_117","patches":7892,"coordinate_shape":[7892,2],"level_dim":[97280,212992],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_118","patches":4143,"coordinate_shape":[4143,2],"level_dim":[97792,218112],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_119","patches":3042,"coordinate_shape":[3042,2],"level_dim":[98816,199168],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_120","patches":19903,"coordinate_shape":[19903,2],"level_dim":[163840,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_121","patches":4541,"coordinate_shape":[4541,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_122","patches":7033,"coordinate_shape":[7033,2],"level_dim":[86016,86016],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_123","patches":3423,"coordinate_shape":[3423,2],"level_dim":[96768,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_124","patches":1918,"coordinate_shape":[1918,2],"level_dim":[96256,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_125","patches":14335,"coordinate_shape":[14335,2],"level_dim":[106496,93184],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_126","patches":9318,"coordinate_shape":[9318,2],"level_dim":[97792,221184],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_127","patches":30884,"coordinate_shape":[30884,2],"level_dim":[196608,100352],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_128","patches":5401,"coordinate_shape":[5401,2],"level_dim":[53248,68096],"patch_size_px":566,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_129","patches":2546,"coordinate_shape":[2546,2],"level_dim":[97792,214528],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0},{"slide_id":"test_130","patches":5758,"coordinate_shape":[5758,2],"level_dim":[97792,217088],"patch_size_px":527,"patch_level":0,"out_of_bounds_coordinate_rows":0}]}


In [ ]:
%run /kaggle/working/feature_assignment_uni_official_test.py


In [ ]:
from pathlib import Path
import json
import pandas as pd
output = Path("/kaggle/working/feature-assignment-uni-official-test")
summary = json.loads((output / "test_summary.json").read_text())
display(pd.DataFrame([{k: v for k, v in summary.items() if k not in ("checkpoint_sha256", "confusion_matrix")}]).T.rename(columns={0: "value"}))
display(pd.DataFrame([summary["confusion_matrix"]]))
display(pd.read_csv(output / "test_predictions_ensemble.csv").head())
print("Saved:", *sorted(str(p) for p in output.iterdir()), sep="\n  ")
